# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [22]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import json
load_dotenv(override=True)

True

In [23]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [24]:
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url= os.getenv('OPENROUTER_BASE_URL')
openai = OpenAI(base_url=openrouter_base_url,api_key=openrouter_api_key )

In [26]:
# Some lists!

checklist = []
completed = []

In [27]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [7]:
get_checklist_report()

''

In [28]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [29]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [30]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [14]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [31]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [32]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [33]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [34]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [35]:
def loop(messages):
    response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="openrouter/free", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [36]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [37]:
checklist, completed = [], []
loop(messages)

Checklist #1: Determine the distance between Boston and New York (estimate if not given).
Checklist #2: Define variables: let t be time after 2:00 pm when they meet (in hours).
Checklist #3: Write distance traveled by Train A (Boston) = 60 * t.
Checklist #4: Write distance traveled by Train B (NY) = 80 * (t - 1) since it starts at 3:00 pm (one hour later).
Checklist #5: Set up equation: distance_A + distance_B = total distance between cities.
Checklist #6: Solve for t.
Checklist #7: Convert t to clock time and provide answer.

Estimated distance between Boston and New York ≈ 200 miles (typical rail distance ~215 miles; using 200 for 
simplicity).

Checklist #1: Determine the distance between Boston and New York (estimate if not given).
Checklist #2: Define variables: let t be time after 2:00 pm when they meet (in hours).
Checklist #3: Write distance traveled by Train A (Boston) = 60 * t.
Checklist #4: Write distance traveled by Train B (NY) = 80 * (t - 1) since it starts at 3:00 pm (one hour later).
Checklist #5: Set up equation: distance_A + distance_B = total distance between cities.
Checklist #6: Solve for t.
Checklist #7: Convert t to clock time and provide answer.

Defined t = hours after 2:00 pm when the two trains meet.

Checklist #1: Determine the distance between Boston and New York (estimate if not given).
Checklist #2: Define variables: let t be time after 2:00 pm when they meet (in hours).
Checklist #3: Write distance traveled by Train A (Boston) = 60 * t.
Checklist #4: Write distance traveled by Train B (NY) = 80 * (t - 1) since it starts at 3:00 pm (one hour later).
Checklist #5: Set up equation: distance_A + distance_B = total distance between cities.
Checklist #6: Solve for t.
Checklist #7: Convert t to clock time and provide answer.

...

Solving: 60t + 80(t - 1) = 200 → 60t + 80t - 80 = 200 → 140t = 280 → t = 2 hours.

Checklist #1: Determine the distance between Boston and New York (estimate if not given).
Checklist #2: Define variables: let t be time after 2:00 pm when they meet (in hours).
Checklist #3: Write distance traveled by Train A (Boston) = 60 * t.
Checklist #4: Write distance traveled by Train B (NY) = 80 * (t - 1) since it starts at 3:00 pm (one hour later).
Checklist #5: Set up equation: distance_A + distance_B = total distance between cities.
Checklist #6: Solve for t.
Checklist #7: Convert t to clock time and provide answer.

t = 2 hours after 2:00 pm, so meeting time is 4:00 pm.

Checklist #1: Determine the distance between Boston and New York (estimate if not given).
Checklist #2: Define variables: let t be time after 2:00 pm when they meet (in hours).
Checklist #3: Write distance traveled by Train A (Boston) = 60 * t.
Checklist #4: Write distance traveled by Train B (NY) = 80 * (t - 1) since it starts at 3:00 pm (one hour later).
Checklist #5: Set up equation: distance_A + distance_B = total distance between cities.
Checklist #6: Solve for t.
Checklist #7: Convert t to clock time and provide answer.

Solution: The estimated distance between Boston and New York is about 200 miles.  
Let t be the time in hours after 2:00 pm when the two trains meet.  
- The Boston train (leaving at 2:00 pm) travels 60 t miles.  
- The New York train (leaving at 3:00 pm) travels 80 (t − 1) miles (starts one hour later).  

Setting the sum of the distances equal to the total distance:

[
60t + 80(t-1) = 200
\]

[
60t + 80t - 80 = 200 \;\Longrightarrow\; 140t = 280 \;\Longrightarrow\; t = 2\ \text{hours}
\]

Thus the trains meet 2 hours after 2:00 pm, which is **4:00 pm**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>